In [49]:
# Imports
import math
import random
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

In [50]:
# Seed
set_seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [51]:
# Load data
jokes_df = pd.read_csv("data.csv").drop(columns=["ID"])
jokes_df.head()

,Joke
0,What did the bartender say to the jumper cable...
1,Don't you hate jokes about German sausage? The...
2,Two artists had an art contest... It ended in ...
3,Why did the chicken cross the playground? To g...
4,What gun do you use to hunt a moose? A moosecut!


In [52]:
# Split
dataset = Dataset.from_pandas(jokes_df, preserve_index=False)
dataset = dataset.train_test_split(test_size=0.2, seed=42, shuffle=True)

train = dataset["train"]
test = dataset["test"]

print(len(train), len(test))

1297 325


In [53]:
# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [54]:
# Tokenize
def tokenize(batch):
    return tokenizer(batch["Joke"], truncation=True)

In [55]:
# Apply tokenization
tokenized_train = train.map(tokenize, batched=True, remove_columns=train.column_names)
tokenized_test = test.map(tokenize, batched=True, remove_columns=test.column_names)

Map: 100%|██████████| 325/325 [00:00<00:00, 11619.96 examples/s]


In [56]:
# Group text
block_size = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [57]:
# Build blocks
blocked_tokenized_train = tokenized_train.map(group_texts, batched=True)
blocked_tokenized_test = tokenized_test.map(group_texts, batched=True)

print(len(blocked_tokenized_train), len(blocked_tokenized_test))

Map: 100%|██████████| 325/325 [00:00<00:00, 19241.56 examples/s]

200 51


In [58]:
# Collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [59]:
# Baseline model
model_v1 = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")
model_v1.config.pad_token_id = tokenizer.eos_token_id

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7660.64it/s]


In [60]:
# Baseline trainer
training_args_v1 = TrainingArguments(
    output_dir="./gpt2-jokes-v1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=8,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
)

trainer_v1 = Trainer(
    model=model_v1,
    args=training_args_v1,
    train_dataset=blocked_tokenized_train,
    eval_dataset=blocked_tokenized_test,
    data_collator=data_collator,
)

In [ ]:
# Train v1
trainer_v1.train()

Epoch,Training Loss,Validation Loss


In [ ]:
# Helpers
def show_results(trainer, label, eval_dataset):
    if trainer.state.global_step == 0:
        print(f"{label} has not been trained in this session. Run the training cell first.")
        return None

    results = trainer.evaluate(eval_dataset=eval_dataset)
    print(label, results)

    if "eval_loss" in results:
        try:
            print(label, "perplexity:", math.exp(results["eval_loss"]))
        except OverflowError:
            print(label, "perplexity: overflow")

    return results

def plot_losses(trainer, title):
    log_history = trainer.state.log_history

    train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
    eval_logs = [x for x in log_history if "eval_loss" in x]

    train_steps = [x["step"] for x in train_logs]
    train_losses = [x["loss"] for x in train_logs]

    eval_steps = [x["step"] for x in eval_logs]
    eval_losses = [x["eval_loss"] for x in eval_logs]

    plt.figure(figsize=(8, 5))
    plt.plot(train_steps, train_losses, label="train_loss")
    plt.plot(eval_steps, eval_losses, label="eval_loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.show()

def generate_joke(model, prompt, max_new_tokens=60):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.1,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# V1 eval
eval_results_v1 = show_results(trainer_v1, "v1", blocked_tokenized_test)

if eval_results_v1 is not None:
    plot_losses(trainer_v1, "GPT-2 v1 Loss")

RuntimeError: on_train_begin must be called before on_evaluate

In [ ]:
# V1 generations
dataset_prompt = " ".join(train[0]["Joke"].split()[:3])
random_prompt = "penguins love pizza"

print("Dataset prompt:", dataset_prompt)
print(generate_joke(trainer_v1.model, dataset_prompt, max_new_tokens=60))

print("\nRandom prompt:", random_prompt)
print(generate_joke(trainer_v1.model, random_prompt, max_new_tokens=60))

In [ ]:
# Fresh v2
model_v2 = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")
model_v2.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
# V2 trainer
training_args_v2 = TrainingArguments(
    output_dir="./gpt2-jokes-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    num_train_epochs=15,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
)

trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=blocked_tokenized_train,
    eval_dataset=blocked_tokenized_test,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
c:\Users\jarre\OneDrive\Documents\GitHub\ICS435Assignment4\.env\Lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=23) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


what did the chicken say to the chicken? "I'm going to kill you." ~Skip to the next page


In [ ]:
# Train v2
trainer_v2.train()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


So this guy is a little bit of a joke. He's a little bit of a joke. He's a


c:\Users\jarre\OneDrive\Documents\GitHub\ICS435Assignment4\.env\Lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=23) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [ ]:
# V2 eval
eval_results_v2 = show_results(trainer_v2, "v2", blocked_tokenized_test)

if eval_results_v2 is not None:
    plot_losses(trainer_v2, "GPT-2 v2 Loss")

In [ ]:
# Compare
results_df = pd.DataFrame(
    {
        "model": ["v1", "v2"],
        "eval_loss": [eval_results_v1["eval_loss"], eval_results_v2["eval_loss"]],
        "perplexity": [
            math.exp(eval_results_v1["eval_loss"]),
            math.exp(eval_results_v2["eval_loss"]),
        ],
    }
).sort_values("eval_loss")

results_df

In [ ]:
# V2 generations
print("Dataset prompt:", dataset_prompt)
print(generate_joke(trainer_v2.model, dataset_prompt, max_new_tokens=60))

print("\nRandom prompt:", random_prompt)
print(generate_joke(trainer_v2.model, random_prompt, max_new_tokens=60))

In [ ]:
# V1 loss plot
log_history_v1 = trainer_v1.state.log_history

train_logs_v1 = [x for x in log_history_v1 if "loss" in x and "eval_loss" not in x]
eval_logs_v1 = [x for x in log_history_v1 if "eval_loss" in x]

train_steps_v1 = [x["step"] for x in train_logs_v1]
train_losses_v1 = [x["loss"] for x in train_logs_v1]

eval_steps_v1 = [x["step"] for x in eval_logs_v1]
eval_losses_v1 = [x["eval_loss"] for x in eval_logs_v1]

plt.figure(figsize=(8, 5))
plt.plot(train_steps_v1, train_losses_v1, label="train_loss")
plt.plot(eval_steps_v1, eval_losses_v1, label="eval_loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("GPT-2 v1 Loss")
plt.legend()
plt.show()

In [ ]:
# V2 loss plot
log_history_v2 = trainer_v2.state.log_history

train_logs_v2 = [x for x in log_history_v2 if "loss" in x and "eval_loss" not in x]
eval_logs_v2 = [x for x in log_history_v2 if "eval_loss" in x]

train_steps_v2 = [x["step"] for x in train_logs_v2]
train_losses_v2 = [x["loss"] for x in train_logs_v2]

eval_steps_v2 = [x["step"] for x in eval_logs_v2]
eval_losses_v2 = [x["eval_loss"] for x in eval_logs_v2]

plt.figure(figsize=(8, 5))
plt.plot(train_steps_v2, train_losses_v2, label="train_loss")
plt.plot(eval_steps_v2, eval_losses_v2, label="eval_loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("GPT-2 v2 Loss")
plt.legend()
plt.show()

In [ ]:
# Combined plot
plt.figure(figsize=(8, 5))
plt.plot(eval_steps_v1, eval_losses_v1, label="v1 eval_loss")
plt.plot(eval_steps_v2, eval_losses_v2, label="v2 eval_loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("GPT-2 v1 vs v2 Eval Loss")
plt.legend()
plt.show()